# 🌙 NYXARA — Google Colab me chalao (step by step)

Ye notebook **NYXARA** ko Google Colab par chalane ke liye hai. Coding aani zaroori nahi — bas har cell ko **upar se neeche, ek-ek karke** run karo.

**Cell run kaise karte hain?** Cell ke left side wale ▶️ (play) button par click karo, ya cell select karke `Shift + Enter` dabao.

| Step | Kya hoga | Kitna time |
|------|----------|-----------|
| 0 | System check (GPU mila ya nahi) | 5 sec |
| 1 | Code download (GitHub token chahiye) | 30 sec |
| 2 | Install + Colab ke liye auto-config | 2–5 min |
| 3 | *(Optional)* Google Drive — memory + model hamesha ke liye save | 1 min |
| 4 | NYXARA se baat karo! 🎉 | — |

> **⚡ Pehle ye karo:** Upar menu me **Runtime → Change runtime type → T4 GPU → Save**. NYXARA ka LLM brain (DistilGPT-2, ~82M) CPU par bhi aaraam se chal jaata hai — chhota aur fast. GPU zaroori nahi.


## Step 0 — System check 🔍

Ye cell batayega ki GPU mila ya nahi, aur sab kuch theek hai ya nahi. Kuch install nahi karta — bas check karta hai.


In [ ]:
import os, shutil, sys

print("🔍 System check...\n")

v = sys.version_info
if (v.major, v.minor) >= (3, 11):
    print(f"✅ Python {v.major}.{v.minor} — theek hai")
else:
    print(f"❌ Python {v.major}.{v.minor} — NYXARA ko 3.11+ chahiye (Colab me normally hota hai)")

try:
    import torch
    GPU = torch.cuda.is_available()
except Exception:
    GPU = False

if GPU:
    print(f"✅ GPU mil gaya: {torch.cuda.get_device_name(0)} 🚀")
else:
    print("⚠️  GPU nahi mila — CPU par bhi chalegi, bas thodi slow rahegi.")
    print("   Fast karne ke liye: Runtime → Change runtime type → T4 GPU → Save,")
    print("   phir cells dobara upar se run karo.")

ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
disk_gb = shutil.disk_usage("/content").free / 1e9
print(f"✅ RAM: {ram_gb:.0f} GB  |  Disk free: {disk_gb:.0f} GB")
print("\n👍 Ab Step 1 par jao.")


## Step 1 — Code download karo 📦

Repo **private** hai, isliye GitHub **Personal Access Token** chahiye. Banane ka tarika (sirf pehli baar):

1. [github.com](https://github.com) par login karo
2. Right-top profile photo → **Settings** → sabse neeche **Developer settings**
3. **Personal access tokens → Tokens (classic)** → **Generate new token (classic)**
4. Koi bhi naam do, **`repo`** wale checkbox par tick karo → **Generate token**
5. Jo `ghp_...` se shuru hone wala token dikhe, use copy kar lo

**🔑 Pro tip (recommended — token baar-baar paste nahi karna padega):**
Colab ke left sidebar me **🔑 (key) icon** par click karo → **Add new secret** → Name me `GITHUB_TOKEN` likho, Value me apna token paste karo, aur **Notebook access** toggle ON karo. Bas — ab ye cell token khud utha lega, har baar paste nahi karna padega.

Secret nahi banaya to koi baat nahi — cell khud token maangega, paste kar ke Enter dabao (token screen par dikhega nahi, ye normal hai).


In [ ]:
import os
from getpass import getpass

REPO = "nyxarajp-del/NYXARAv01"
DEST = "/content/NYXARAv01"

def _get_token():
    # Pehle Colab Secrets (🔑) me GITHUB_TOKEN dhundo, warna paste karne ko bolo
    try:
        from google.colab import userdata
        t = userdata.get("GITHUB_TOKEN")
        if t and t.strip():
            print("🔑 Token Colab Secrets se mil gaya!")
            return t.strip()
    except Exception:
        pass
    return getpass("Apna GitHub token paste karo aur Enter dabao: ").strip()

if os.path.isdir(f"{DEST}/.git"):
    # Code pehle se hai — bas latest update kheench lo
    %cd {DEST}
    token = _get_token()
    !git pull https://{token}@github.com/{REPO}.git
    print("\n✅ Code update ho gaya (latest version)!")
else:
    token = _get_token()
    !git clone https://{token}@github.com/{REPO}.git {DEST}
    assert os.path.isdir(f"{DEST}/.git"), (
        "❌ Download fail hua — upar ka error padho. Zyadatar wajah: token galat hai "
        "ya usme `repo` scope tick nahi kiya. Naya token banao aur cell dobara run karo."
    )
    %cd {DEST}
    # Token ko git config me mat chhodo (safety)
    !git remote set-url origin https://github.com/{REPO}.git
    print("\n✅ Code download ho gaya!")

del token  # token memory me mat rakho


## Step 2 — Install + auto-config ⚙️

Ye cell:
- NYXARA aur uski saari libraries install karega (**2–5 minute** — Colab me torch/transformers pehle se hote hain, isliye utna lamba nahi lagta)
- NYXARA ki settings **khud set** karega (DistilGPT-2 chhota model hai — CPU par bhi full-precision me aaraam se chalta hai)
- End me check karega ki sab theek install hua

Beech me kuch pip warnings dikhein to **ghabrao mat, wo normal hai**. Jab tak `✅ Install complete!` na dikhe, wait karo.


In [ ]:
%cd /content/NYXARAv01

# 1) NYXARA + core libraries (reasoning, LLM, memory, security, self-training)
!pip install -q -e ".[reasoning,llm,vector,security,observe,foundry]" httpx "pydantic>=2.6" "pydantic-settings>=2.1"

# 2) DistilGPT-2 (~82M) chhota hai — CPU par full-precision me hi chalta hai, quantization ki zaroorat nahi
import torch
GPU = torch.cuda.is_available()

# 3) Colab ke hisaab se NYXARA ka config (.env) likho
env_config = """# NYXARA — Colab auto-config (Step 2 ne banaya)
NYXARA_PROFILE=dev
NYXARA_LLM__PROVIDER=auto
NYXARA_LLM__QWEN_MODEL=distilgpt2
NYXARA_LLM__QWEN_LOAD_IN_4BIT=false
"""
with open(".env", "w", encoding="utf-8") as f:
    f.write(env_config)

# 4) Sanity check — import ho raha hai?
import nyxara  # noqa: F401
print(f"\n✅ Install complete!  (device: {'GPU 🚀' if GPU else 'CPU'} — DistilGPT-2 dono par fast chalta hai)")


## Step 3 *(Optional, par recommended)* — Google Drive jodo 💾

Colab session band hote hi sab kuch delete ho jaata hai. Ye cell do cheezein bachata hai:

1. **NYXARA ki memory** — wo aapko yaad rakhegi, uski learning agli baar bhi bachi rahegi
2. **LLM model ka download (~350 MB)** — agli baar dobara download nahi karna padega, seedha Drive se load hoga

Sab kuch aapke Google Drive ke `NYXARA_memory` folder me jaayega. Permission popup aaye to apna Google account choose kar ke **Allow** karo.

*(Nahi chahiye to skip karo — seedha Step 4 par jao.)*


In [ ]:
import os, shutil
from google.colab import drive

drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/NYXARA_memory"
os.makedirs(BASE, exist_ok=True)

def _link(local_path, drive_dir):
    """local_path ko Drive ke folder par point kara do (purana data move karke)."""
    os.makedirs(drive_dir, exist_ok=True)
    local_path = os.path.expanduser(local_path)
    if os.path.islink(local_path):
        os.unlink(local_path)
    elif os.path.isdir(local_path):
        for name in os.listdir(local_path):
            src, dst = os.path.join(local_path, name), os.path.join(drive_dir, name)
            if not os.path.exists(dst):
                shutil.move(src, dst)
        shutil.rmtree(local_path, ignore_errors=True)
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    os.symlink(drive_dir, local_path)
    return drive_dir

# 1) NYXARA ki memory (~/.nyxara) → Drive
#    (purane notebook wale users ka data seedha NYXARA_memory me tha — use wahi rehne do)
brain_dir = BASE if os.path.exists(os.path.join(BASE, "memory.json")) else os.path.join(BASE, "brain")
print("✅ Memory ab Drive me save hogi:", _link("~/.nyxara", brain_dir))

# 2) Model cache → Drive (agli baar DistilGPT-2 dobara download nahi hoga)
print("✅ Model cache bhi Drive me:", _link("~/.cache/huggingface", os.path.join(BASE, "model_cache")))


## Step 4 — NYXARA se baat karo! 🎉

Ye cell NYXARA ka console start karega.

- **Pehli baar** boot me thoda time lagega — LLM model (~350 MB) download hota hai. Agar Step 3 me Drive joda tha, to agli baar se download skip ho jayega.
- `Master>` dikhne par box me apna message type karo aur Enter dabao
- Commands ke liye `/help` type karo — jaise `/report` (status), `/wander` (usay sochne do), `/learning` (learning state), `/explain` (pichla jawab kyun aisa tha)
- **Band karne ke liye `/quit` type karo** — quit par NYXARA apni memory save kar leti hai

**Note:** Ye cell tab tak chalta rahega jab tak aap `/quit` nahi karte — ye **normal hai, error nahi**. Cell ke neeche input box me hi type karna hai.


In [ ]:
%cd /content/NYXARAv01

print("⏳ NYXARA boot ho rahi hai... (pehli baar DistilGPT-2 download hota hai — ~1 min)\n")

from nyxara.__main__ import main
main()


## Agli baar kya karna hai? 🔁

Colab session band hone par install delete ho jaata hai (Drive wala data bacha rehta hai). Agli baar bas:

1. Ye notebook phir kholo — Colab me **File → Open notebook → GitHub** tab, **"Include private repos"** tick karo, aur `nyxarajp-del/NYXARAv01` search karo
2. **Runtime → Change runtime type → T4 GPU** select karo
3. Step 0 se Step 4 tak cells dobara run karo (Colab Secrets me token saved hai to kuch paste bhi nahi karna padega)

Agar Step 3 me Drive joda tha, to NYXARA ki **purani memory wapas load hogi — wo aapko yaad rakhegi** 🌙 aur model bhi dobara download nahi hoga.

---

## Kuch gadbad ho to (Troubleshooting) 🔧

| Problem | Solution |
|---------|----------|
| `Your session crashed after using all available RAM` | DistilGPT-2 bahut chhota hai, RAM shayad hi kam padegi. Runtime restart karke saare cells dobara run karo. |
| Step 1 me `Authentication failed` / `Repository not found` | Token galat ya expire hai, ya `repo` scope tick nahi kiya. Naya token banao, Colab Secret update karo, cell dobara run karo. |
| Runtime disconnect ho gaya / "Restart runtime" aa gaya | Kuch kharab nahi hua — Step 0 se saare cells dobara run kar do. |
| Step 4 wala cell rukta hi nahi | Wo **normal** hai — console chal raha hai. Band karne ke liye input box me `/quit` likho. |
| Pehli baar boot thoda slow | Model download ho raha hai (~350 MB) — ek baar ka kaam hai. Step 3 (Drive) use karoge to dobara kabhi nahi hoga. |
| Drive me jagah nahi (`No space left`) | Google Drive me kam se kam ~3 GB free chahiye. Kuch files delete karo ya Step 3 skip kar do. |

Aur koi dikkat ho to error message copy kar ke ChatGPT/Claude se pooch lo, ya repo me issue khol do. 🌙
